# Day 2 — LangGraph: Stateful Agent Workflows

---

Yesterday's hand-rolled loop works for one linear flow. Real agents need:

- **Branching** — "if search returned nothing, try a different query"
- **Loops with clear exit conditions**
- **Resumability** — pause for human input, then continue
- **Debuggability** — see exactly what each step did, replay a failed run
- **State** — carry findings, sources, retries, timestamps across steps

That's what **LangGraph** gives you. It's a small library from the LangChain team that models an agent as a **graph of nodes** with typed **state** flowing between them.

Not the only option — LlamaIndex Workflows and Pydantic AI are close alternatives. **LangGraph is the most common in 2026 job listings**, so it's what we teach.


## 1. The mental model — graph, state, nodes, edges

Four concepts, that's the whole abstraction:

- **State** — a Python dict (usually a `TypedDict`) that flows through the graph
- **Node** — a function that takes state, returns a partial state update
- **Edge** — connects one node to the next; can be **conditional**
- **Graph** — the whole thing, compiled into an executor

```
       ┌────────┐
       │  plan  │  \
       └────┬───┘   │
            │       │
            ▼       │
       ┌────────┐   │ conditional edge:
       │ search │   │ if answer is good  -> done
       └────┬───┘   │ else               -> retry
            │       │
            ▼       │
       ┌────────┐   │
       │  done  │  /
       └────────┘
```

Each box is a node. Arrows are edges. Some edges branch based on state.

**Why this shape?** Because every non-trivial agent looks like this — steps that pass information forward, sometimes looping, sometimes branching. LangGraph makes the graph explicit instead of hiding it in a while-loop with a lot of `if`s.


## 2. Setup


In [ ]:
!pip install langgraph together python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from together import Together
llm = Together()
MODEL = "openai/gpt-oss-20b"


## 3. State — deeper than you think

State is a Python `TypedDict`. Each field is a slot that nodes read and write.

```python
class AgentState(TypedDict):
    question: str
    plan: str
    steps_taken: Annotated[list[str], operator.add]
    answer: str
```

The **`Annotated[list, operator.add]`** part is a LangGraph superpower called a **reducer**. It tells the graph: *"when multiple nodes update this field, don't overwrite — combine them."*

Without a reducer, if two nodes both return `{"steps_taken": ["x"]}`, the last one wins. With `operator.add` as the reducer, both get appended and you end up with `["x", "x"]`.

Common reducers:

| Reducer | What it does | Use for |
|---|---|---|
| *(none)* | Overwrite (last write wins) | Simple string / int fields |
| `operator.add` | Append to a list | Logs, step history, findings |
| `add_messages` (from langgraph.graph.message) | Append messages the LangChain way | Chat history |
| Custom function `(old, new) -> merged` | Anything you want | Merge dicts, dedupe, max, etc. |

For beginners: **`operator.add` covers 90% of cases.**


## 4. Define the state


In [ ]:
from typing import TypedDict, Annotated
import operator

class AgentState(TypedDict):
    question: str
    plan: str
    steps_taken: Annotated[list[str], operator.add]
    answer: str


## 5. Define the nodes

A node is a plain Python function. It:
1. Takes **the current state** as its only argument
2. Returns a **dict of updates** (partial state — LangGraph merges it in for you)

Nothing magic. No decorators required, no special base class.


In [ ]:
def plan_node(state: AgentState) -> dict:
    prompt = f"Question: {state['question']}\nWrite a 1-2 sentence plan for how to answer this."
    resp = llm.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.0,
    )
    return {"plan": resp.choices[0].message.content.strip(),
            "steps_taken": ["planned"]}


def act_node(state: AgentState) -> dict:
    prompt = (f"Question: {state['question']}\nPlan: {state['plan']}\n"
              "Now write a concise final answer.")
    resp = llm.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.2,
    )
    return {"answer": resp.choices[0].message.content.strip(),
            "steps_taken": ["acted"]}


def review_node(state: AgentState) -> dict:
    prompt = (f"Question: {state['question']}\nAnswer: {state['answer']}\n"
              "Rewrite the answer to be crisp and under 50 words.")
    resp = llm.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.0,
    )
    return {"answer": resp.choices[0].message.content.strip(),
            "steps_taken": ["reviewed"]}


**Two things worth noticing:**

- Every node returns a `dict` with the fields it wants to update — not the whole state. LangGraph merges automatically.
- `steps_taken` uses the `operator.add` reducer, so each node's `["planned"]` / `["acted"]` gets **appended** to the running list, not overwritten. By the end, `steps_taken == ["planned", "acted", "reviewed"]`.


## 6. Wire the graph


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)
builder.add_node("plan",   plan_node)
builder.add_node("act",    act_node)
builder.add_node("review", review_node)

builder.add_edge(START,    "plan")
builder.add_edge("plan",   "act")
builder.add_edge("act",    "review")
builder.add_edge("review", END)

graph = builder.compile()


**Two special nodes** you always use:

- `START` — the graph's entry point. Every graph needs an edge from `START`.
- `END` — a sink node. Once a node's edge points to `END`, execution stops.

`compile()` returns an executor you can invoke, stream, or inspect.


## 7. Run it


In [ ]:
final = graph.invoke({"question": "Explain HTTPS in one paragraph.",
                       "plan": "", "steps_taken": [], "answer": ""})

print("Steps :", final["steps_taken"])
print("\nPlan  :", final["plan"])
print("\nAnswer:", final["answer"])


**Note the initial state** — you pass a starting dict. All fields you'll write to should exist (with sensible empty values). LangGraph is strict about the schema.

For a linear 3-step flow this feels like overkill (and it is). **The win comes with conditional edges.**


## 8. Conditional edges — the real reason to use a framework

Say we want to **retry** if the answer looks weak. Add a **conditional edge**: a function that inspects state and returns the *name* of the next node.


In [ ]:
def is_answer_ok(state: AgentState) -> str:
    # Toy check: reject one-line answers
    return "end" if len(state["answer"]) > 40 else "retry"


builder2 = StateGraph(AgentState)
builder2.add_node("plan",   plan_node)
builder2.add_node("act",    act_node)
builder2.add_node("review", review_node)
builder2.add_edge(START,  "plan")
builder2.add_edge("plan", "act")
builder2.add_conditional_edges(
    "act",
    is_answer_ok,
    {"end": "review", "retry": "plan"},
)
builder2.add_edge("review", END)
graph2 = builder2.compile()

r = graph2.invoke({"question": "What is 2+2?",
                   "plan": "", "steps_taken": [], "answer": ""})
print("Steps :", r["steps_taken"])
print("Answer:", r["answer"])


**How `add_conditional_edges` works:**

- First arg: the source node
- Second arg: a router function `(state) -> str` — returns a **key**
- Third arg: a dict mapping key → target node

If `is_answer_ok` returns `"retry"`, control flows to `"plan"` (loop!). If it returns `"end"`, it flows to `"review"`.

LangGraph handles the loop, cycle detection, and state accumulation. Try hand-writing this correctly with retries + max-iterations + state accumulation — that's 100+ lines. LangGraph made it 5.


## 9. `create_react_agent` — the batteries-included shortcut

If your agent is a straightforward "loop while there are tool calls" flow (like yesterday's), LangGraph ships a prebuilt.


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_together import ChatTogether     # LangChain wrapper for Together


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


# create_react_agent expects a LangChain-shaped chat model
model = ChatTogether(model="openai/gpt-oss-20b", temperature=0)
agent = create_react_agent(model, [add, multiply])

result = agent.invoke({"messages": [("user", "What is (3+5) * 4?")]})
for m in result["messages"]:
    m.pretty_print()


`create_react_agent` gave you:
- Tool schemas auto-generated from `@tool`-decorated functions + docstrings
- The full tool-calling loop
- Message history in the state
- Streaming support out of the box

Under the hood it's exactly the graph you'd hand-wire: `agent` node → tool calls → `tools` node → back to `agent`. It just saved you 50 lines.

Use it when your agent is "LLM + tools, loop until done." Roll your own graph when you need custom nodes (plan/execute/review/HITL).


## 10. Checkpointing — resume across invocations

For long-running agents (especially HITL — Day 4), you need to **pause** and **resume**. LangGraph does this via **checkpointers**: state gets persisted after every node.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()               # in-memory; swap for SqliteSaver in prod
graph_ckpt = builder2.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "user-42-session-1"}}

# First call — runs to completion or first interrupt
r = graph_ckpt.invoke({"question": "What is 2+2?", "plan": "", "steps_taken": [], "answer": ""},
                      config=config)
print("Steps so far:", r["steps_taken"])

# Later — inspect state, resume, or fork under a different thread_id
snapshot = graph_ckpt.get_state(config)
print("Current node:", snapshot.next)      # what would run next


**Checkpointers you'll see in real projects:**

| Checkpointer | When |
|---|---|
| `MemorySaver` | Tests, notebooks — dies on restart |
| `SqliteSaver` | Small production, local file — durable, single-node |
| `PostgresSaver` | Multi-worker production |
| `RedisSaver` | High-throughput / distributed |

**`thread_id`** is how LangGraph knows which conversation / task a request belongs to. Two users with different `thread_id`s get completely isolated state — no leakage.


## 11. Streaming — show progress as it happens

`graph.invoke(...)` waits for the whole graph to finish. `graph.stream(...)` yields **state updates after each node**, which is perfect for a UI showing "planning...", "searching...", "writing...".


In [ ]:
for event in graph.stream({"question": "Explain HTTPS in one paragraph.",
                            "plan": "", "steps_taken": [], "answer": ""}):
    # event is {node_name: state_update_dict}
    for node, update in event.items():
        print(f"[{node}] -> keys updated: {list(update.keys())}")


There's also `graph.stream(..., stream_mode="values")` for full-state snapshots, `"messages"` for token-by-token LLM streaming inside nodes, and `"debug"` for verbose tracing. Read the docs when you need each.


## 12. Debugging with LangGraph Studio

For any non-trivial graph, install **LangGraph Studio** (desktop app). It gives you a visual node-graph, live state inspector, breakpoint support, and thread-timeline view. It's the LLM equivalent of a debugger — worth learning early.

```bash
# One-time install:
pip install "langgraph[all]"
# Then point it at any compiled graph via a langgraph.json manifest.
```


## 13. Multi-agent — AutoGen, CrewAI (one paragraph)

**AutoGen** (Microsoft) and **CrewAI** are libraries for orchestrating **multiple agents talking to each other** — e.g. a "researcher" + "writer" + "critic". LangGraph can do this too (each agent is a node; edges route messages between them).

**Fresher advice:** don't reach for multi-agent unless there's a *concrete* reason (distinct role prompts, non-overlapping tool sets, or an actual review-and-approve dynamic). Most "multi-agent" problems are better solved as one agent with a well-designed LangGraph. Know the names, save the deep dive for a real project.


## Recap

- LangGraph models an agent as **state + nodes + edges**.
- **State** = TypedDict; use **`Annotated[..., operator.add]`** to accumulate lists.
- **Nodes** = plain functions returning partial state updates.
- **Conditional edges** are the killer feature — clean branches and loops.
- `create_react_agent` = the batteries-included shortcut for pure tool-loop agents.
- **Checkpointers** persist state across invocations — the foundation for HITL and resumability.
- `graph.stream(...)` gives you per-node progress events for UIs.
- Multi-agent (AutoGen / CrewAI) exists; use sparingly.
- **Next class:** memory — short-term (this conversation) vs long-term (past sessions in a vector store).
